# 00 — Experiment dashboard

Read-only view over `experiments/log.csv`, `submissions/*/meta.json` and the newest run log. No pipeline
logic here — this notebook only reads files the pipeline (`src/run_pipeline.py`) already writes.
Safe to run any time, including while a pipeline run is in progress in another process.


In [ ]:
import pathlib, sys

NB_DIR = pathlib.Path.cwd().resolve()
_d = NB_DIR
while not (_d / "CLAUDE.md").exists():
    if _d == _d.parent:
        raise RuntimeError("could not find student_resource/ (looked for a CLAUDE.md marker)")
    _d = _d.parent
REPO_ROOT = _d
PKG_DIR = REPO_ROOT / "code" / "business_entity_resolution"
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))
DATA_DIR = REPO_ROOT / "dataset"
OUT_DIR = REPO_ROOT / "output"
print("repo root:", REPO_ROOT)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## a) All experiments (experiments/log.csv)

In [ ]:
log_path = REPO_ROOT / "experiments" / "log.csv"
if log_path.exists():
    log_df = pd.read_csv(log_path)
    display(log_df)
else:
    log_df = pd.DataFrame()
    print(f"not found: {log_path}")


## b) Dev F0.5 by experiment (all / US / India / singletons)

In [ ]:
score_cols = ["f05_all", "f05_us", "f05_in", "f05_singletons"]
plot_df = log_df.dropna(subset=["f05_all"]) if not log_df.empty else pd.DataFrame()
if plot_df.empty:
    print("no experiment rows with a dev F0.5 score yet")
else:
    ax = plot_df.set_index("exp_id")[score_cols].plot(marker="o", figsize=(8, 4), title="dev F0.5 by exp_id")
    ax.set_ylabel("F0.5")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## c) Blocking recall and candidates/entity by experiment

In [ ]:
block_df = log_df.dropna(subset=["block_recall"]) if not log_df.empty else pd.DataFrame()
if block_df.empty:
    print("no experiment rows with blocking stats yet")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.bar(block_df["exp_id"], block_df["block_recall"])
    ax1.set_title("block_recall")
    ax1.axhline(0.97, color="red", linestyle="--", linewidth=1, label="target 0.97")
    ax1.legend()
    ax1.tick_params(axis="x", rotation=45)
    ax2.bar(block_df["exp_id"], block_df["cands_per_entity"])
    ax2.set_title("cands_per_entity")
    ax2.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


## d) Uploads: public LB per submission, uploads used per day (limit 5)

In [ ]:
sub_dir = REPO_ROOT / "submissions"
metas = []
if sub_dir.exists():
    for p in sorted(sub_dir.glob("*/meta.json")):
        try:
            metas.append(json.loads(p.read_text()))
        except (json.JSONDecodeError, OSError) as e:
            print(f"skipping {p}: {e}")

if not metas:
    print(f"no submissions archived yet under {sub_dir} (run scripts/archive_submission.py after an upload)")
else:
    sub_df = pd.DataFrame(metas)
    sub_df["public_lb"] = pd.to_numeric(sub_df["public_lb"], errors="coerce")
    sub_df["day"] = pd.to_datetime(sub_df["archived_at"], errors="coerce").dt.date
    display(sub_df[["tag", "exp_id", "public_lb", "archived_at", "commit"]])

    counts = sub_df.groupby("day").size()
    print("uploads per day (limit 5):")
    for day, n in counts.items():
        flag = "  <-- AT/OVER LIMIT" if n >= 5 else ""
        print(f"  {day}: {n}{flag}")

    scored = sub_df.dropna(subset=["public_lb"])
    if not scored.empty:
        scored.plot.bar(x="tag", y="public_lb", figsize=(8, 4), legend=False, title="public LB by submission")
        plt.tight_layout()
        plt.show()


## e) Newest run log (last 15 lines)

In [ ]:
run_logs = sorted(REPO_ROOT.glob("*run*.log"), key=lambda p: p.stat().st_mtime, reverse=True)
if not run_logs:
    print("no *run*.log found under", REPO_ROOT)
else:
    newest = run_logs[0]
    print(f"newest: {newest.name} (modified {pd.Timestamp(newest.stat().st_mtime, unit='s')})")
    lines = newest.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-15:]))


## f) Python processes currently running (optional — needs psutil)

In [ ]:
try:
    import psutil
    procs = [(p.pid, " ".join(p.info["cmdline"] or []))
             for p in psutil.process_iter(["pid", "name", "cmdline"])
             if p.info["name"] and "python" in p.info["name"].lower()]
    if procs:
        for pid, cmd in procs:
            print(f"  pid={pid}  {cmd[:140]}")
    else:
        print("no python processes found")
except ImportError:
    print("psutil not installed; skipping process list (pip install psutil, or check manually: "
          "Get-Process python / ps aux | grep python)")
